### Cargar Red

In [ ]:
import pypsa
n = #red a analizar
n1 = #red con capex existente

INFO:pypsa.io:Imported network elec_s_9_ec_ll1.25_1H_demand.nc has buses, carriers, generators, lines, links, loads, storage_units, stores
INFO:pypsa.io:Imported network elec_s_9_ec_lv1_1H_demand.nc has buses, carriers, generators, global_constraints, lines, loads, storage_units


### Valores Generales de Flujo Energético

In [ ]:
import pandas as pd

# Use objective weighting if available (best for energy sums)
wcol = "objective" if "objective" in n.snapshot_weightings.columns else "generators"
w = n.snapshot_weightings[wcol]

# Helper function to convert to TWh
def to_TWh(series_or_df):
    """Apply snapshot weighting and convert MW to TWh"""
    return (series_or_df.mul(w, axis=0)).sum().sum() / 1e6

# ============================================================================
# 1. LOAD SHEDDING
# ============================================================================
shed_mask = (n.generators.carrier == "load shedding")
load_shedding_TWh = to_TWh(n.generators_t.p.loc[:, shed_mask])

# ============================================================================
# 2. GENERATED POWER (exclude load shedding, include reservoir from storage)
# ============================================================================
gen_mask = ~shed_mask
generated_power_TWh = to_TWh(n.generators_t.p.loc[:, gen_mask])

# Add reservoir hydro from storage units
if not n.storage_units.empty:
    hydro_mask = n.storage_units.carrier == "hydro"
    if hydro_mask.any():
        hydro_discharge = n.storage_units_t.p.loc[:, hydro_mask].clip(lower=0.0)
        generated_power_TWh += to_TWh(hydro_discharge)

# ============================================================================
# 3. PUMPED HYDRO STORAGE (PHS) - StorageUnits
# ============================================================================
phs_charge_TWh = 0.0
phs_discharge_TWh = 0.0

if not n.storage_units.empty:
    phs_mask = n.storage_units.carrier == "PHS"
    if phs_mask.any():
        phs_discharge = n.storage_units_t.p.loc[:, phs_mask].clip(lower=0.0)
        phs_discharge_TWh = to_TWh(phs_discharge)
        
        phs_charge = (-n.storage_units_t.p.loc[:, phs_mask].clip(upper=0.0))
        phs_charge_TWh = to_TWh(phs_charge)

# ============================================================================
# 4. BATTERIES - StorageUnits
# ============================================================================
battery_charge_TWh = 0.0
battery_discharge_TWh = 0.0

if not n.storage_units.empty:
    battery_mask = n.storage_units.carrier == "battery"
    if battery_mask.any():
        battery_discharge = n.storage_units_t.p.loc[:, battery_mask].clip(lower=0.0)
        battery_discharge_TWh = to_TWh(battery_discharge)
        
        battery_charge = (-n.storage_units_t.p.loc[:, battery_mask].clip(upper=0.0))
        battery_charge_TWh = to_TWh(battery_charge)

# ============================================================================
# 5. HYDROGEN (Store + Links)
# ============================================================================
h2_charge_TWh = 0.0
h2_discharge_TWh = 0.0

h2_chg_links = n.links.index[n.links.carrier == "H2 electrolysis"]
h2_dis_links = n.links.index[n.links.carrier == "H2 fuel cell"]

# Charging power is on p0 for electrolysis (from AC bus)
if len(h2_chg_links) > 0:
    h2_charge_MW = n.links_t.p0[h2_chg_links].clip(lower=0.0)
    h2_charge_TWh = to_TWh(h2_charge_MW)

# Discharging power: use p1 at the AC bus side (bus1). Sign can be negative.
if len(h2_dis_links) > 0:
    p1 = n.links_t.p1[h2_dis_links]

    if p1.sum().sum() < 0:
        h2_discharge_MW = (-p1).clip(lower=0.0)
    else:
        h2_discharge_MW = p1.clip(lower=0.0)

    h2_discharge_TWh = to_TWh(h2_discharge_MW)


# ============================================================================
# 6. SERVED DEMAND (excluding charging)
# ============================================================================
demand_TWh = to_TWh(n.loads_t.p_set)
served_demand_TWh = demand_TWh - load_shedding_TWh

# ============================================================================
# OUTPUT - Clean table format
# ============================================================================
results = pd.DataFrame({
    'Metric': [
        'Load Shedding',
        'Generated Power',
        'PHS Charge',
        'PHS Discharge',
        'Battery Charge',
        'Battery Discharge',
        'H2 Charge',
        'H2 Discharge',
        'Served Demand'
    ],
    'TWh': [
        load_shedding_TWh,
        generated_power_TWh,
        phs_charge_TWh,
        phs_discharge_TWh,
        battery_charge_TWh,
        battery_discharge_TWh,
        h2_charge_TWh,
        h2_discharge_TWh,
        served_demand_TWh
    ]
})

charge = phs_charge_TWh + battery_charge_TWh + h2_charge_TWh
discharge = phs_discharge_TWh + battery_discharge_TWh + h2_discharge_TWh

pd.options.display.float_format = '{:.2f}'.format
print(results.to_string(index=False))
print(f"Charge: {charge:,.2f} TWh")
print(f"Discharge: {discharge:,.2f} TWh")

           Metric    TWh
    Load Shedding   0.00
  Generated Power 258.00
       PHS Charge   1.75
    PHS Discharge   1.40
   Battery Charge  80.64
Battery Discharge  74.32
        H2 Charge  11.56
     H2 Discharge   3.59
    Served Demand 243.36
Charge: 93.95 TWh
Discharge: 79.32 TWh


### Curtailment

In [97]:
p = n.generators_t.p

p_avail = n.generators_t.p_max_pu.multiply(
    n.generators.p_nom_opt.fillna(n.generators.p_nom),
    axis=1
)

vre = n.generators.carrier.isin(["solar", "onwind"])
p = p.loc[:, vre]
p_avail = p_avail.loc[:, vre]

curtailment = p_avail - p

dt = n.snapshot_weightings["objective"]

curtailment_energy = (curtailment.mul(dt, axis=0)).sum().sum()
available_energy = (p_avail.mul(dt, axis=0)).sum().sum()

curtailment_share = (curtailment_energy / available_energy)*100

print(f"Available energy: {(available_energy)/1000000:,.2f} TWh")
print(f"Curtailment energy: {(curtailment_energy)/1000:,.2f} GWh")
print(f"Curtailment share: {curtailment_share:.2f} %")



INFO:pypsa.io:Imported network elec_s_9_ec_ll1.25_1H_demand.nc has buses, carriers, generators, lines, loads, storage_units


Available energy: 154.69 TWh
Curtailment energy: 13,143.65 GWh
Curtailment share: 8.50 %


### Costos

In [73]:

h = n.snapshot_weightings.generators  # hours per snapshot

# ── Energy ───────────────────────────────────────────────────────────────────
demand_MWh = (n.loads_t.p_set.sum(axis=1) * h).sum()

if "load shedding" in n.generators.carrier.values:
    ls_idx = n.generators.index[n.generators.carrier == "load shedding"]
    shed_MWh = (n.generators_t.p[ls_idx].sum(axis=1).clip(lower=0.0) * h).sum()
else:
    shed_MWh = 0.0

served_MWh = demand_MWh - shed_MWh

# ── Existing Infrastructure CAPEX ─────────────────────────────────────────────

# $/MWh values for existing infrastructure — edit these
CAPEX_RATE = {
    "hydro":   50.0,  # applies to reservoir, ror, and PHS
    "ocgt":    200.0,
    "ccgt":    78.0,
    "nuclear": 180.0,
    "solar":   58.0,
    "onwind":  61.0,
    "lines":   3.0,
}

def get_annual_generation_MWh(carrier_list):
    """Sum of MWh produced across all generators with carrier in carrier_list."""
    idx = n1.generators.index[n1.generators.carrier.isin(carrier_list)]
    gen_MWh = (n1.generators_t.p[idx].mul(h, axis=0)).sum().sum() if len(idx) else 0.0

    # Also check storage units (PHS dispatches as storage_unit)
    su_idx = n1.storage_units.index[n1.storage_units.carrier.isin(carrier_list)]
    su_MWh = (n1.storage_units_t.p[su_idx].clip(lower=0.0).mul(h, axis=0)).sum().sum() if len(su_idx) else 0.0

    return gen_MWh + su_MWh

def get_opex_for_carriers(carrier_list):
    """OPEX contribution (USD/year) from generators with carrier in carrier_list."""
    idx = n1.generators.index[n1.generators.carrier.isin(carrier_list)]
    if len(idx) == 0:
        return 0.0
    return (
        n1.generators_t.p[idx]
        .mul(h, axis=0)
        .mul(n1.generators.loc[idx, "marginal_cost"], axis=1)
        .sum().sum()
    )

# Annual generation per technology group (MWh)
hydro_carriers = ["hydro", "ror", "PHS", "phs"]  # adjust carrier names to match your network
gen_MWh = {
    "hydro":   get_annual_generation_MWh(hydro_carriers),
    "ocgt":    get_annual_generation_MWh(["ocgt", "OCGT"]),
    "ccgt":    get_annual_generation_MWh(["ccgt", "CCGT"]),
    "nuclear": get_annual_generation_MWh(["nuclear"]),
    "solar":   get_annual_generation_MWh(["solar"]),
    "onwind":  get_annual_generation_MWh(["onwind"]),
}

# OCGT + CCGT OPEX (to subtract from their CAPEX)
ocgt_ccgt_opex = (
    get_opex_for_carriers(["ocgt", "OCGT"]) +
    get_opex_for_carriers(["ccgt", "CCGT"])
)

# Total line flow (MWh) — sum of |p0| across all lines and snapshots
total_line_flow_MWh = n1.lines_t.p0.abs().mul(h, axis=0).sum().sum()

# CAPEX calculation
capex_existing = 0.0
for tech, mwh in gen_MWh.items():
    contribution = mwh * CAPEX_RATE[tech]
    if tech in ("ocgt", "ccgt"):
        contribution -= ocgt_ccgt_opex if tech == "ocgt" else 0.0  # subtract once, not twice
    capex_existing += contribution

capex_existing += total_line_flow_MWh * CAPEX_RATE["lines"]

# ── CAPEX ─────────────────────────────────────────────────────────────────────
capex = 0.0

for component, p_nom_attr in [
    ("Generator",    "p_nom_opt"),
    ("StorageUnit",  "p_nom_opt"),
    ("Link",         "p_nom_opt"),
    ("Line",         "s_nom_opt"),
]:
    df = n.df(component)
    ext = df[df.get("p_nom_extendable", df.get("s_nom_extendable", False)) == True]
    if ext.empty:
        continue
    capex += (ext.capital_cost * (ext[p_nom_attr] - ext[p_nom_attr.replace("opt", "min")])).sum()

# ── OPEX ─────────────────────────────────────────────────────────────────────
opex = 0.0

gens = n.generators.query("carrier != 'load shedding'")
opex += (
    n.generators_t.p[gens.index]
    .mul(h, axis=0)
    .mul(gens.marginal_cost, axis=1)
    .sum().sum()
)

opex += (
    n.storage_units_t.p[n.storage_units.index].clip(lower=0.0)
    .mul(h, axis=0)
    .mul(n.storage_units.marginal_cost, axis=1)
    .sum().sum()
)

opex += (
    n.links_t.p0[n.links.index].abs()
    .mul(h, axis=0)
    .mul(n.links.marginal_cost, axis=1)
    .sum().sum()
)

# ── Congestion rent ───────────────────────────────────────────────────────────
lam = n.buses_t.marginal_price

line_rent = (
    lam[n.lines.bus0.values].set_axis(n.lines.index, axis=1) -
    lam[n.lines.bus1.values].set_axis(n.lines.index, axis=1)
) * n.lines_t.p0

total_congestion_rent = line_rent.mul(h, axis=0).sum(axis=0).abs().sum()

# ── Load shedding penalty ─────────────────────────────────────────────────────
ls = n.generators.query("carrier == 'load shedding'")
penalty_cost = (
    n.generators_t.p[ls.index]
    .mul(ls.marginal_cost, axis=1)
    .mul(h, axis=0)
    .sum().sum()
)

# ── Results ───────────────────────────────────────────────────────────────────
total_system_cost = n.objective  # fuel + variable O&M + fixed O&M + CAPEX + shedding penalty
total_system_cost += capex_existing + total_congestion_rent  # add existing CAPEX and congestion rent to get full system cost
precio_monomico_pypsa = total_system_cost / served_MWh
total_capex = capex_existing + capex

print(f"Total System Cost:              ${total_system_cost:,.2f}")
print(f"Existing Infrastructure CAPEX:  ${capex_existing:,.2f}")
print(f"New CAPEX:                      ${capex:,.2f}")
print(f"Total CAPEX:                    ${total_capex:,.2f}")
print(f"OPEX:                           ${opex:,.2f}")
print(f"Load Shedding Penalty:          ${penalty_cost:,.2f}")
print(f"Total Congestion Rent:          ${total_congestion_rent:,.2f}")
print(f"Precio Monómico PyPSA:          ${precio_monomico_pypsa:,.2f} per MWh")

Total System Cost:              $17,776,020,359.59
Existing Infrastructure CAPEX:  $6,600,309,945.35
New CAPEX:                      $4,278,959,423.82
Total CAPEX:                    $10,879,269,369.17
OPEX:                           $6,364,294,848.25
Load Shedding Penalty:          $124,554,704.25
Total Congestion Rent:          $407,901,437.84
Precio Monómico PyPSA:          $73.05 per MWh


### Tabla por Carrier

In [ ]:
import pandas as pd

def compute_opex_table(n: object):
    h = n.snapshot_weightings.generators

    def _energy_and_opex(p, marginal_cost, carrier):
        p_pos    = p.clip(lower=0)
        weighted = p_pos.mul(h, axis=0)
        energy   = weighted.sum().groupby(carrier).sum()
        opex     = weighted.mul(marginal_cost, axis=1).sum().groupby(carrier).sum()
        return energy, opex

    # generators (exclude load shedding)
    g = n.generators.query("carrier != 'load shedding'")
    g_E, g_O = _energy_and_opex(n.generators_t.p[g.index], g.marginal_cost, g.carrier)

    # storage units
    su = n.storage_units
    su_E, su_O = _energy_and_opex(n.storage_units_t.p[su.index], su.marginal_cost, su.carrier)

    # H2 fuel cells (p1 is per MWh input, clip negative discharge side)
    h2 = n.links.query("carrier == 'H2 fuel cell'")
    if not h2.empty:
        p1 = n.links_t.p1[h2.index].clip(upper=0).abs()
        h2_E, h2_O = _energy_and_opex(p1, h2.marginal_cost, h2.carrier)
    else:
        h2_E = h2_O = pd.Series(dtype=float)

    E  = pd.concat([g_E,  su_E,  h2_E]).groupby(level=0).sum()
    O  = pd.concat([g_O,  su_O,  h2_O]).groupby(level=0).sum()
    mc = pd.concat([g[["carrier", "marginal_cost"]],
                    su[["carrier", "marginal_cost"]],
                    *([] if h2.empty else [h2[["carrier", "marginal_cost"]]])
                   ]).groupby("carrier")["marginal_cost"].mean()

    df = pd.DataFrame({
        "OPEX_USD_per_year":     O,
        "Energy_MWh":            E,
        "Marginal_Cost_USD_MWh": mc,
    }).sort_values("OPEX_USD_per_year", ascending=False)

    totals = df[["OPEX_USD_per_year", "Energy_MWh"]].sum()
    totals["Marginal_Cost_USD_MWh"] = float("nan")
    df.loc["TOTAL"] = totals

    return df.style.format("{:,.2f}", na_rep="-")


compute_opex_table(n)

INFO:pypsa.io:Imported network elec_s_9_ec_lv1_1H_demand.nc has buses, carriers, generators, global_constraints, lines, loads, storage_units


,OPEX_USD_per_year,Energy_MWh,Marginal_Cost_USD_MWh
CCGT,"5,011,128,201.50","76,893,605.83",65.17
nuclear,"246,113,801.18","15,373,797.75",16.01
OCGT,"16,625,800.07","186,245.89",89.27
biomass,"7,141,864.99","315,359.29",22.65
onwind,"337,187.73","13,570,718.42",0.02
ror,"247,052.94","23,346,035.60",0.01
hydro,"225,381.17","8,839,799.50",0.03
solar,"59,106.34","2,374,002.39",0.02
PHS,631.08,"67,009.79",0.01
TOTAL,"5,281,879,027.00","140,966,574.45",-
